In [4]:
import kagglehub
import os

# Download dataset
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Dataset Path:", path)

# List files inside folder
files = os.listdir(path)

print("\nFiles inside dataset:")
for f in files:
    print(f)

Dataset Path: C:\Users\bhara\.cache\kagglehub\datasets\manishkr1754\cardekho-used-car-data\versions\2

Files inside dataset:
cardekho_dataset.csv


In [5]:
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Dataset Path:", path)

# Find CSV automatically
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

print("CSV Files Found:", csv_files)

# Load first CSV found
file_path = os.path.join(path, csv_files[0])

df = pd.read_csv(file_path)

# Fix column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

print("\nLoaded Successfully!")
print("Shape:", df.shape)

df.head()

Dataset Path: C:\Users\bhara\.cache\kagglehub\datasets\manishkr1754\cardekho-used-car-data\versions\2
CSV Files Found: ['cardekho_dataset.csv']

Loaded Successfully!
Shape: (15411, 14)


,Unnamed:_0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [8]:
import kagglehub
import pandas as pd
import numpy as np
import os

# Download dataset
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Dataset Path:", path)

# Automatically find CSV
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

file_path = os.path.join(path, csv_files[0])

# Load dataset
df = pd.read_csv(file_path)

# Fix column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Inspect dataset
print("Initial Shape:", df.shape)

print("\nDataset Info:")
print(df.info())

print("\nMissing %:")
print(df.isnull().sum() * 100 / len(df))


# Drop rows where selling_price missing
df = df.dropna(subset=["selling_price"])


# Clean mileage, engine, max_power
cols = ["mileage", "engine", "max_power"]

for col in cols:
    df[col] = df[col].astype(str).str.extract('([0-9.]+)')
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].median(), inplace=True)


# Remove unrealistic prices
df = df[
    (df["selling_price"] != 999999999) &
    (df["selling_price"] >= 10000)
]


# Drop duplicates
df = df.drop_duplicates()


# Drop unwanted index column
if "Unnamed:_0" in df.columns:
    df = df.drop(columns=["Unnamed:_0"])


print("\nFinal Shape After Cleaning:", df.shape)

Dataset Path: C:\Users\bhara\.cache\kagglehub\datasets\manishkr1754\cardekho-used-car-data\versions\2
Initial Shape: (15411, 14)

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed:_0         15411 non-null  int64  
 1   car_name           15411 non-null  str    
 2   brand              15411 non-null  str    
 3   model              15411 non-null  str    
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  str    
 7   fuel_type          15411 non-null  str    
 8   transmission_type  15411 non-null  str    
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null

C:\Users\bhara\AppData\Local\Temp\ipykernel_17308\764114097.py:42: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(df[col].median(), inplace=True)
C:\Users\bhara\AppData\Local\Temp\ipykernel_17308\764114097.py:42: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment u

In [9]:
# Label Encoding transmission_type
df["transmission_type"] = df["transmission_type"].map({
    "Manual": 0,
    "Automatic": 1
})


# One-hot encoding fuel_type and seller_type
df = pd.get_dummies(
    df,
    columns=["fuel_type", "seller_type"],
    drop_first=True
)


# Print final columns
print("\nFinal Columns:")
print(df.columns.tolist())


Final Columns:
['car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


# Define X and y
X = df.drop(columns=["selling_price"])

y = df["selling_price"]


# Drop remaining non-numeric columns
X = X.select_dtypes(include=["number"])


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Baseline prediction
baseline_pred = [y_train.mean()] * len(y_test)


# Calculate MAE
mae = mean_absolute_error(y_test, baseline_pred)


print(f"Baseline MAE: ₹{round(mae)}")

Baseline MAE: ₹468748
